# Solving the Taxi Fare Problem with Linear Regression

Welcome! Instead of just looking at abstract math, let's solve a real-world problem: **How much should a taxi ride cost?**

We will walk through the 4 core steps of Linear Regression and see how each one helps us reach the answer.

---

## 🚕 The Problem: Predicting the Price of a Ride

Imagine you are starting a new taxi company. You need a fair way to charge customers. You know that the further someone travels, the more they should pay, but there's also a base fee just for getting in the car.

**Linear Regression helps us solve this by finding the "Best Fit Line" that represents the relationship between Distance and Price.**

## 📂 Step 1: Data Gathering
**How it helps:** You can't guess a fair price without looking at what has happened in the past. We collect data on hundreds of previous trips (Miles traveled and Fare paid).

**In this notebook:** We will use a dataset of Chicago Taxi trips to find these patterns.

## 📏 Step 2: Defining the Model (The Mathematical Blueprint)
**How it helps:** We need a structure for our prediction. We assume the price follows a straight line:

**$$y = wx + b$$**

- **y (Fare)**: What we want to solve for.
- **x (Miles)**: The input we know.
- **w (Weight/Rate)**: How much we charge per mile.
- **b (Bias/Base Fee)**: The starting price of the ride.

Linear Regression's job is to find the **perfect values** for $w$ and $b$.

## 📉 Step 3: Measuring Error (The Loss Function)
**How it helps:** If we guess a rate of $5/mile, but the real data shows people usually pay $3/mile, we are "wrong." The **Loss Function** (Mean Squared Error) gives us a single number that tells us exactly *how* wrong we are.

**The Logic:**
1. Calculate the difference: $(Actual - Predicted)$
2. Square it: $(Actual - Predicted)^2$ (This punishes big mistakes more than small ones).
3. Average it: This gives us the **Mean Squared Error (MSE)**.

**Formula:**
$$MSE = \frac{1}{N} \sum (y_{actual} - y_{predicted})^2$$

## 🔄 Step 4: Improving the Model (Gradient Descent)
**How it helps:** Now that we know we are wrong, which way do we change our rate? Do we make it higher or lower? 

**Gradient Descent** uses calculus to find the "slope" of our error. It tells us exactly which direction to move our weight ($w$) to make the error smaller.

### Deriving the "Fix":
1. **Start with Loss**: $Loss = (Actual - (w \cdot Miles + b))^2$
2. **Find the Slope (Derivative)**: Using the chain rule, we find how the loss changes when $w$ changes:
   - Slope = $2 \cdot (Actual - Predicted) \cdot (-Miles)$
3. **Update the Weight**: We subtract a small piece of this slope from our current weight.

**$$w_{new} = w_{old} - (Learning\_Rate \times Gradient)$$**

In [ ]:
# Manual Step-by-Step Solve
weight = 10.0      # Our initial "bad" guess: $10 per mile
learning_rate = 0.05 

actual_fare = 25.0 # A real trip that cost $25
miles = 2.0        # That trip was 2 miles long

print(f"Initial Guess ($10/mile): ${weight * miles}")

# Step 3: Measure Error
error = (weight * miles) - actual_fare # 20 - 25 = -5

# Step 4: Find Gradient and Update
gradient = miles * error # Simplified gradient
weight = weight - (learning_rate * gradient)

print(f"New Rate after learning: ${weight:.2f}/mile")
print(f"New Prediction: ${weight * miles:.2f} (Much closer to $25!)")

---
## 🛠️ Applying this to Thousands of Trips

Now that we understand the steps, let's use **TensorFlow** to perform these 4 steps on the full Chicago Taxi dataset.

In [ ]:
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

# Step 1: Gathering Data
DATA_URL = 'https://download.mlcc.google.com/mledu-datasets/chicago_taxi_train.csv'
df = pd.read_csv(DATA_URL)
training_df = df.loc[:, ('TRIP_MILES', 'TRIP_SECONDS', 'FARE')].dropna()

print(f"Loaded {len(training_df)} trips to learn from.")
training_df.head()

### Defining our Automated Solver

- `tf.keras.layers.Dense(units=1)`: This creates our **Mathematical Blueprint** ($y=wx+b$).
- `optimizer=RMSprop`: This is our **Gradient Descent** engine. It handles all the complex math to update the weights automatically.

In [ ]:
def build_model(learning_rate):
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(units=1, input_shape=(1,))
    ])
    model.compile(
        optimizer=tf.keras.optimizers.RMSprop(learning_rate=learning_rate),
        loss='mean_squared_error',
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )
    return model

def train_model(model, df, feature, label, epochs, batch_size):
    history = model.fit(x=df[feature], y=df[label], batch_size=batch_size, epochs=epochs, verbose=0)
    return history

## 🧪 Running Experiments

Let's see the model learn in real-time. As the **Epochs** (passes through the data) increase, notice how the **Error (RMSE)** goes down. This is Gradient Descent in action!

In [ ]:
learning_rate = 0.01
epochs = 20
batch_size = 50

model = build_model(learning_rate)
history = train_model(model, training_df, 'TRIP_MILES', 'FARE', epochs, batch_size)

plt.plot(history.history['root_mean_squared_error'])
plt.title('The Learning Process: Error decreasing as Model improves')
plt.xlabel('Training Steps (Epochs)')
plt.ylabel('Error in Dollars ($)')
plt.show()